<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/08_compare_detectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 · Compare detectors (zero-training)

How well do the detectors find our 20 classes **without any fine-tuning**? Scores three
backends from `src/detector.py` on the merged dataset's **test** split:

| backend | what it runs |
|---|---|
| COCO-only | `yolo11n.pt`: only the COCO classes (person, car, bench, ...) |
| openvocab | YOLOE `yoloe-11s-seg.pt` prompted with `configs/prompts.yaml` (all 20 classes) |
| hybrid | yolo11n for COCO classes + YOLOE for the other 10 classes, merged with NMS |

```
reads   MyDrive/clear_vision/datasets/merged.zip + merged.done   (from notebook 02)
writes  /content/datasets/merged_test/          only images/test, labels/test, data.yaml
        MyDrive/clear_vision/runs/compare_detectors/<date>.json
```

- **Output:** one table, class | COCO-only mAP50 | openvocab mAP50 | hybrid mAP50, with the
  rare classes (curb, stairs, pothole, manhole, bollard, barrier, traffic_cone, pole,
  scooter, crosswalk) highlighted.
- **Only the test files are extracted** from the 5.7 GB zip, with `zipfile`.
- **How it scores:**
  - Thresholds are lowered to `EVAL_CONF` and the open-vocab model runs on every image.
  - The AP is COCO-style 101-point AP at IoU 0.5, computed by
    `scripts/compare_detectors.py`. Hybrid is not one Ultralytics model, so `model.val()`
    can't score it.
  - "-" means the backend cannot output that class (COCO-only on the rare classes).
- **Runtime:** a GPU is recommended (Runtime → Change runtime type → T4 GPU). On CPU it
  works but is slow. The FPS in section 4 is GPU speed; the Mac numbers are in
  `configs/default.yaml`.
- The first run downloads YOLOE and its text encoder (~600 MB) to `models/` in the repo
  clone. Run all cells.

In [ ]:
# Settings
REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
TEST_DEST = "/content/datasets/merged_test"  # extracted test split (Colab disk)
BACKENDS = ["ultralytics", "openvocab", "hybrid"]  # ultralytics = COCO-only
EVAL_CONF = 0.01  # detection threshold while scoring mAP
MAX_IMAGES = None  # e.g. 300 for a quick run; None = the whole test split
FPS_IMAGES = 200  # test images used for the speed table in section 4

## 1. Setup

In [ ]:
# 1. Setup: Drive, repo, packages
import logging
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = Path("/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics "git+https://github.com/ultralytics/CLIP.git"

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import yaml  # noqa: E402

import compare_detectors as cd  # noqa: E402
from src.detector import load_class_names  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU: this works but is slow. Runtime -> Change runtime type -> T4 GPU.")

MERGED_ZIP = Path(MERGED_ZIP)
DET_CFG = yaml.safe_load((REPO_DIR / "configs" / "default.yaml").read_text())["detector"]
CLASS_NAMES = load_class_names(REPO_DIR / "configs" / "classes.yaml")
print(f"{len(CLASS_NAMES)} classes: {', '.join(CLASS_NAMES)}")

## 2. Test split

Extracts only `images/test/`, `labels/test/` and `data.yaml` from `merged.zip` (needs
`merged.done`, so the zip is a verified one) and writes a `data.yaml` with our 20 class
names. A second run in the same runtime reuses the extraction.

In [ ]:
# 2. Extract the test split from merged.zip
done = MERGED_ZIP.with_suffix(".done")
if not (MERGED_ZIP.exists() and done.exists()):
    raise RuntimeError(f"Need {MERGED_ZIP} and {done.name}: run notebook 02 first.")

DATA_YAML = cd.extract_test_split(MERGED_ZIP, Path(TEST_DEST), CLASS_NAMES)
PAIRS = cd.load_split(DATA_YAML, "test", MAX_IMAGES)
boxes = sum(len(cd.yolo_label_boxes(label)) for _, label in PAIRS)
print(f"data.yaml: {DATA_YAML}")
print(f"test images: {len(PAIRS)}   boxes: {boxes}")

## 3. mAP50 per class

Builds each backend (the first open-vocab build prompts YOLOE once and caches the prompted
model in `models/openvocab/`) and scores it on the test split.

In [ ]:
# 3. mAP50 per class for each backend
import math

import pandas as pd

MAPS = cd.run_eval(DET_CFG, BACKENDS, DATA_YAML, CLASS_NAMES, EVAL_CONF, MAX_IMAGES)

columns = [f"{cd.LABELS[b]} mAP50" for b in BACKENDS]
rows = {c: [MAPS[b][c] for b in BACKENDS] for c in CLASS_NAMES}
rows["mean (all)"] = [cd.mean_map(MAPS[b], CLASS_NAMES) for b in BACKENDS]
rows["mean (rare)"] = [cd.mean_map(MAPS[b], [c for c in CLASS_NAMES if c in cd.RARE_CLASSES]) for b in BACKENDS]
table = pd.DataFrame.from_dict(rows, orient="index", columns=columns, dtype=object)
table.index.name = "class"


def fmt(v: object) -> str:
    if v is None:
        return "-"  # this backend cannot output the class
    return "n/a" if math.isnan(v) else f"{v:.3f}"  # n/a = no test boxes of this class


def highlight(row: pd.Series) -> list[str]:
    if row.name in cd.RARE_CLASSES:
        return ["background-color: #fff3cd; color: #000; font-weight: bold"] * len(row)
    if str(row.name).startswith("mean"):
        return ["border-top: 1px solid #888; font-style: italic"] * len(row)
    return [""] * len(row)


print(cd.format_map_table(MAPS, CLASS_NAMES))
display(table.style.apply(highlight, axis=1).format(fmt).set_caption(
    "mAP50 on the TEST split, no fine-tuning (highlighted = rare classes)"))

## 4. Speed and detections per class

Runs each backend as it runs live (default thresholds, hybrid's open-vocab model every
`openvocab_every` frames) on the first `FPS_IMAGES` test images. These FPS are GPU numbers,
not the Mac's.

In [ ]:
# 4. FPS and detection counts with the live settings
from src.detector import build_detector

frames = [f for _, f in cd.iter_frames([p for p, _ in PAIRS[:FPS_IMAGES]])]
SPEED = {b: cd.benchmark(build_detector(DET_CFG, b), frames) for b in BACKENDS}
print(cd.format_counts_table(SPEED))

## 5. Save results

In [ ]:
# 5. Save the tables to Drive
import json
from datetime import datetime

out = Path(DRIVE_ROOT) / "runs" / "compare_detectors" / f"{datetime.now():%Y%m%d_%H%M}.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({
    "commit": commit.strip(),
    "test_images": len(PAIRS),
    "eval_conf": EVAL_CONF,
    "map50": {b: {c: (None if v is None or math.isnan(v) else v) for c, v in MAPS[b].items()} for b in BACKENDS},
    "speed": SPEED,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))
print(f"Saved {out}")